# Band power — Castellar replication (Flow paper, active only)

Replicates Núñez Castellar et al. (2019, *Psychophysiology*) Study 2.
Computes band power from response-locked CSD epochs/evoked.

**Active only** — passive participants have no response events.

Two methods:
- **`castellar_exact`**: single FFT on trial-averaged CSD evoked, cropped 0–250ms
- **`castellar_trial`**: single FFT per trial, then average power across trials

Castellar parameters:
- Frequency resolution: 3.125Hz (n_fft=328 at 1024Hz)
- Window: boxcar (no taper), detrended
- Time window: 0–250ms post-response
- Bands: theta (3.5–8.5Hz), alpha (8.5–12.5Hz), beta (12.5–30Hz)
- Electrodes: Fz, FCz, Cz

Outputs (one per band, matching R stats script naming):
- `{date}_active_{n}p_flow_CastellarBand-{band}_statsnorm.csv`
  - long format, log10-transformed, both methods combined
  - columns: participant, cond, electrode, method, power

Input: step 8 response-locked CSD files (hits only) — identical before and after the
2026-10-30 step 8 fix (verified: r = 1.000, same trial counts), so results should match the
previous run exactly (checked in the last cell).

Last edit: 31.10.2026 @anguyen

In [1]:
from __future__ import annotations

import re
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from mne import read_epochs, read_evokeds
from scipy.fft import rfft, rfftfreq
from scipy.integrate import simpson

def find_repo_root(marker='core') -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not locate repo root (no '{marker}/' found above {Path.cwd()})")

sys.path.insert(0, str(find_repo_root() / 'core'))
from participants import get_participants

DATE_STR = datetime.today().strftime('%Y%m%d')

In [2]:
# ==============================================================================
# Configuration
# ==============================================================================

PAPER    = 'flow'
PARADIGM = 'active'  # hard constraint — passive has no response events

PARTICIPANTS      = get_participants(PAPER, PARADIGM)
participants_kept = list(PARTICIPANTS.keys())
n                 = len(participants_kept)
print(f'Loaded {n} participants')

# Castellar-matching parameters
TARGET_FREQ_RESOLUTION_HZ = 3.125
CASTELLAR_TMIN    = 0.0
CASTELLAR_TMAX    = 0.25
CASTELLAR_BANDS   = {
    'theta': (3.5,  8.5),
    'alpha': (8.5,  12.5),
    'beta':  (12.5, 30.0),
}
CASTELLAR_ELECTRODES = ['Fz', 'FCz', 'Cz']

# group mapping: high/low flow run assignment
high_first = ['G2', 'G4', 'G6', 'G8']
low_first  = ['G1', 'G3', 'G5', 'G7']

if sys.platform == 'win32':
    path        = Path('L:/EEG_Flow_data/Study_EEG_Flow_Active/derivatives_v2')
    output_root = Path('L:/EEG_Flow_data/Study_EEG_Flow_Active/derivatives_v2/outputs/measures/flow/freq/castellar')
else:
    path        = Path('/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_Active/derivatives_v2')
    output_root = Path('/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_Active/derivatives_v2/outputs/measures/flow/freq/castellar')

output_root.mkdir(parents=True, exist_ok=True)
print(f'Output: {output_root}')

Loaded 36 participants
Output: L:\EEG_Flow_data\Study_EEG_Flow_Active\derivatives_v2\outputs\measures\flow\freq\castellar


In [3]:
# ==============================================================================
# Helper functions
# ==============================================================================

def extract_uid(file: str) -> str:
    m = re.search(r'sub-([A-Za-z0-9]+)-', file)
    if m is None:
        raise ValueError(f'Could not extract UID from: {file}')
    return m.group(1)


def get_n_fft(sfreq: float, participant: str, n_times: int, method: str) -> int:
    n_fft      = int(round(sfreq / TARGET_FREQ_RESOLUTION_HZ))
    actual_res = sfreq / n_fft
    if abs(actual_res - TARGET_FREQ_RESOLUTION_HZ) > 0.1:
        print(f'[{method}] {participant}: resolution {actual_res:.3f}Hz (target {TARGET_FREQ_RESOLUTION_HZ}Hz)')
    if n_fft > n_times:
        print(f'[{method}] WARNING {participant}: n_fft={n_fft} > n_times={n_times}')
    return n_fft


def single_fft_psd(data: np.ndarray, sfreq: float, n_fft: int):
    """Detrend, zero-pad, single FFT, one-sided PSD. data: (..., n_times)."""
    data    = data - data.mean(axis=-1, keepdims=True)
    n_in    = data.shape[-1]
    if n_fft > n_in:
        pad = [(0, 0)] * (data.ndim - 1) + [(0, n_fft - n_in)]
        data = np.pad(data, pad)
    spec    = rfft(data, n=n_fft, axis=-1)
    freqs   = rfftfreq(n_fft, d=1.0 / sfreq)
    power   = (np.abs(spec) ** 2) / (sfreq * n_fft)
    power[..., 1:-1] *= 2  # one-sided doubling
    return freqs, power


def run_condition(files, cond_name, process_fn, n_jobs=-1):
    print(f'\n=== {cond_name} ({len(files)} files) ===')
    results = Parallel(n_jobs=n_jobs)(delayed(process_fn)(f) for f in files)
    df = pd.concat([r for r in results if r is not None], ignore_index=True)
    df['cond'] = cond_name
    return df


print('Helpers ready.')

Helpers ready.


In [4]:
# ==============================================================================
# Method castellar_exact: FFT on trial-averaged evoked
# Method castellar_trial: FFT per trial, then average power
# ==============================================================================

def process_evoked_exact(file: str):
    participant = extract_uid(file)
    evokeds     = read_evokeds(file, verbose=False)
    assert len(evokeds) == 1
    evoked      = evokeds[0].copy().crop(tmin=CASTELLAR_TMIN, tmax=CASTELLAR_TMAX)
    sfreq       = evoked.info['sfreq']
    data        = evoked.data
    n_fft       = get_n_fft(sfreq, participant, data.shape[-1], 'castellar_exact')
    freqs, power = single_fft_psd(data, sfreq, n_fft)
    mask        = (freqs >= 1.0) & (freqs <= 40.0)
    df = pd.DataFrame(power[:, mask].T, columns=evoked.ch_names)
    df.insert(0, 'freq',        freqs[mask])
    df.insert(0, 'participant', participant)
    df.insert(0, 'method',      'castellar_exact')
    return df


def process_trial_exact(file: str):
    participant = extract_uid(file)
    epochs      = read_epochs(file, verbose=False).crop(tmin=CASTELLAR_TMIN, tmax=CASTELLAR_TMAX)
    sfreq       = epochs.info['sfreq']
    data        = epochs.get_data(copy=False)  # (n_epochs, n_ch, n_times)
    n_fft       = get_n_fft(sfreq, participant, data.shape[-1], 'castellar_trial')
    freqs, power = single_fft_psd(data, sfreq, n_fft)
    mean_power  = power.mean(axis=0)  # average power across trials
    mask        = (freqs >= 1.0) & (freqs <= 40.0)
    df = pd.DataFrame(mean_power[:, mask].T, columns=epochs.ch_names)
    df.insert(0, 'freq',        freqs[mask])
    df.insert(0, 'participant', participant)
    df.insert(0, 'method',      'castellar_trial')
    return df


print('Processing functions ready.')

Processing functions ready.


In [5]:
# ==============================================================================
# File discovery and condition splitting
# ==============================================================================

# per-run files only (sub-X/sub-X_task-T_run-R/), exact participant match
def _find(suffix):
    return sorted(str(f) for f in path.glob(f'sub-*/sub-*_task-*_run-*/sub-*_task-*_run-*_{suffix}')
                  if extract_uid(f.name) in participants_kept)

ave_files = _find('step8_responselocked_CSD_bc-200-0-ave.fif')
epo_files = _find('step8_responselocked_CSD_bc-200-0-epo.fif')

print(f'Ave files: {len(ave_files)}  Epo files: {len(epo_files)}')


def split_by_condition(files):
    def gid_of(f):
        m = re.search(r'sub-[A-Za-z0-9]+-([A-Z0-9]+)', f)
        return m.group(1) if m else None
    return {
        'oddball_1':    [x for x in files if 'oddball' in x and 'run-1' in x],
        'oddball_2':    [x for x in files if 'oddball' in x and 'run-2' in x],
        'UT_highflow':  ([x for x in files if 'UT' in x and 'run-1' in x and gid_of(x) in high_first] +
                         [x for x in files if 'UT' in x and 'run-2' in x and gid_of(x) in low_first]),
        'UT_lowflow':   ([x for x in files if 'UT' in x and 'run-1' in x and gid_of(x) in low_first] +
                         [x for x in files if 'UT' in x and 'run-2' in x and gid_of(x) in high_first]),
    }

ave_by_cond = split_by_condition(ave_files)
epo_by_cond = split_by_condition(epo_files)
for cond, files in ave_by_cond.items():
    print(f'  {cond}: {len(files)} ave / {len(epo_by_cond[cond])} epo')

Ave files: 142  Epo files: 142
  oddball_1: 35 ave / 35 epo
  oddball_2: 35 ave / 35 epo
  UT_highflow: 36 ave / 36 epo
  UT_lowflow: 36 ave / 36 epo


In [6]:
%%time
# ==============================================================================
# Run both methods
# ==============================================================================

dfs_exact = {cond: run_condition(files, cond, process_evoked_exact)
             for cond, files in ave_by_cond.items() if files}
all_exact = pd.concat(dfs_exact.values(), ignore_index=True)

dfs_trial = {cond: run_condition(files, cond, process_trial_exact)
             for cond, files in epo_by_cond.items() if files}
all_trial = pd.concat(dfs_trial.values(), ignore_index=True)

print(f'\nexact rows: {len(all_exact)}  trial rows: {len(all_trial)}')

# temporary save for diagnostic PSD plot — can delete after plotting
all_exact.to_csv(
    output_root / f'{DATE_STR}_active_{n}p_flow_castellar_exact_all_blocks_tmp.csv',
    index=False
)


=== oddball_1 (35 files) ===

=== oddball_2 (35 files) ===

=== UT_highflow (36 files) ===

=== UT_lowflow (36 files) ===

=== oddball_1 (35 files) ===

=== oddball_2 (35 files) ===

=== UT_highflow (36 files) ===

=== UT_lowflow (36 files) ===

exact rows: 1704  trial rows: 1704
CPU times: total: 844 ms
Wall time: 37.2 s


In [7]:
# ==============================================================================
# Band integration and save stats-ready CSVs
# Naming matches R stats script: {date}_active_{n}p_flow_CastellarBand-{band}_statsnorm.csv
# columns: participant, cond, electrode, method, power (log10-transformed)
# ==============================================================================

def integrate_bands(long_df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    long_df = long_df.copy()
    long_df['freq'] = pd.to_numeric(long_df['freq'], errors='coerce')
    electrodes = [e for e in CASTELLAR_ELECTRODES if e in long_df.columns]
    out = {}
    for band_name, (fmin, fmax) in CASTELLAR_BANDS.items():
        rows = []
        for (participant, cond, method), group in long_df.groupby(
                ['participant', 'cond', 'method']):
            filtered = group[
                (group['freq'] > fmin) & (group['freq'] <= fmax)
            ].sort_values('freq')
            freqs = filtered['freq'].to_numpy()
            data  = filtered[electrodes].to_numpy()
            integrated = simpson(data, freqs, axis=0) if len(freqs) > 1 else data.mean(axis=0)
            rows.append({
                'participant': participant,
                'cond':        cond,
                'method':      method,
                **dict(zip(electrodes, integrated * 1e12))  # V² -> µV²
            })
        out[band_name] = pd.DataFrame(rows)
    return out


# combine both methods and integrate
all_combined = pd.concat([all_exact, all_trial], ignore_index=True)
bands        = integrate_bands(all_combined)

# save: one file per band, long format, log10-transformed
electrodes = [e for e in CASTELLAR_ELECTRODES if e in all_exact.columns]

for band_name, df in bands.items():
    df_long = df.melt(
        id_vars=['participant', 'cond', 'method'],
        value_vars=electrodes,
        var_name='electrode', value_name='power'
    )
    df_long['power'] = np.log10(df_long['power'] + 1e-20)

    fname = output_root / f'{DATE_STR}_active_{n}p_flow_CastellarBand-{band_name}_statsnorm.csv'
    df_long.to_csv(fname, index=False)
    print(f'Saved -> {fname.name}')

print('\nDone.')

Saved -> 20261031_active_36p_flow_CastellarBand-theta_statsnorm.csv
Saved -> 20261031_active_36p_flow_CastellarBand-alpha_statsnorm.csv
Saved -> 20261031_active_36p_flow_CastellarBand-beta_statsnorm.csv

Done.


In [8]:
# ==============================================================================
# Diagnostic plots — PSD and band power distributions
# ==============================================================================

import matplotlib.pyplot as plt
import seaborn as sns
from mne.viz import set_browser_backend

%matplotlib qt
set_browser_backend('qt')

img_root = output_root.parents[3] / 'img' / PAPER / 'freq' / 'castellar'
img_root.mkdir(parents=True, exist_ok=True)

# --- 1. PSD line plot at Castellar electrodes ---
flow_conds  = {'UT_highflow': '#E74C3C', 'UT_lowflow': '#2ECC71'}
labels_map  = {'UT_highflow': 'High Flow', 'UT_lowflow': 'Low Flow'}

# reload long CSVs for PSD plot
psd_dfs = {}
for cond in ['UT_highflow', 'UT_lowflow']:
    f = sorted(output_root.glob(f'*_long_*p_castellar_*all_blocks*.csv'))
    # use the combined all_blocks file
all_exact_df = pd.read_csv(sorted(output_root.glob('*castellar_exact_all_blocks_tmp.csv'))[-1])

fig, axes = plt.subplots(1, len(CASTELLAR_ELECTRODES),
                          figsize=(5 * len(CASTELLAR_ELECTRODES), 5), sharey=True)

for cond, color in flow_conds.items():
    df_cond = all_exact_df[all_exact_df['cond'] == cond]
    for ax, elec in zip(axes, CASTELLAR_ELECTRODES):
        if elec not in df_cond.columns:
            continue
        mean_psd = df_cond.groupby('freq')[elec].mean() * 1e12
        ax.plot(mean_psd.index, mean_psd.values, color=color,
                label=labels_map[cond], lw=2)
        for band, (fmin, fmax) in CASTELLAR_BANDS.items():
            ax.axvspan(fmin, fmax, alpha=0.1, color='gray')
        ax.set_title(elec)
        ax.set_xlabel('Frequency (Hz)')
        ax.set_xlim(1, 35)

axes[0].set_ylabel('Power (µV²/Hz)')
axes[0].legend()
fig.suptitle(f'Castellar PSD — evoked_exact | N={n}')
plt.tight_layout()
plt.savefig(img_root / f'{DATE_STR}_{n}p_flow_castellar_psd.svg', dpi=300, bbox_inches='tight')
plt.show()

# --- 2. Band power distributions ---
stats_files = {band: sorted(output_root.glob(f'*CastellarBand-{band}_statsnorm.csv'))[-1]
               for band in CASTELLAR_BANDS}

fig, axes = plt.subplots(len(CASTELLAR_BANDS), len(CASTELLAR_ELECTRODES),
                          figsize=(5 * len(CASTELLAR_ELECTRODES), 4 * len(CASTELLAR_BANDS)),
                          sharey='row', squeeze=False)

for row, (band_name, fpath) in enumerate(stats_files.items()):
    df = pd.read_csv(fpath)
    df = df[df['cond'].isin(['UT_highflow', 'UT_lowflow'])]
    df['cond_label'] = df['cond'].map({'UT_highflow': 'High Flow', 'UT_lowflow': 'Low Flow'})

    for col, elec in enumerate(CASTELLAR_ELECTRODES):
        ax  = axes[row, col]
        df_elec = df[df['electrode'] == elec]
        sns.boxplot(
            data=df_elec, x='cond_label', y='power',
            hue='method', ax=ax, palette='Set2'
        )
        ax.set_title(f'{band_name} — {elec}')
        ax.set_xlabel('')
        ax.set_ylabel('log10(µV²)' if row == 0 else '')
        if col > 0 or row > 0:
            ax.legend().remove()

fig.suptitle(f'Castellar band power distributions | N={n}', fontsize=14)
plt.tight_layout()
plt.savefig(img_root / f'{DATE_STR}_{n}p_flow_castellar_bandpower_distributions.svg',
            dpi=300, bbox_inches='tight')
plt.show()

Using qt as 2D backend.


In [9]:
# ==============================================================================
# Consistency check — compare with the previous run (response-locked data unchanged)
# ==============================================================================
for band_name in CASTELLAR_BANDS:
    files = sorted(output_root.glob(f'*_active_*p_flow_CastellarBand-{band_name}_statsnorm.csv'))
    new   = [f for f in files if f.name.startswith(DATE_STR)]
    old   = [f for f in files if not f.name.startswith(DATE_STR)]
    if not (new and old):
        print(f'{band_name}: nothing to compare'); continue
    key = ['participant', 'cond', 'method', 'electrode']
    m = pd.read_csv(old[-1]).merge(pd.read_csv(new[-1]), on=key, suffixes=('_old', '_new'))
    d = (m['power_new'] - m['power_old']).abs()
    print(f'{band_name}: {old[-1].name[:8]} vs {new[-1].name[:8]} | {len(m)} rows | '
          f'max |diff| = {d.max():.2e} log10 units -> {"IDENTICAL" if d.max() < 1e-6 else "DIFFERENT"}')

theta: 20261016 vs 20261031 | 852 rows | max |diff| = 0.00e+00 log10 units -> IDENTICAL
alpha: 20261016 vs 20261031 | 852 rows | max |diff| = 0.00e+00 log10 units -> IDENTICAL
beta: 20261016 vs 20261031 | 852 rows | max |diff| = 0.00e+00 log10 units -> IDENTICAL
